# GOTSF 추가 실험 — Colab 재현 노트북 (2) · 최종본

> **최종본**: v1–v7에서 확인한 설정을 모두 반영해, 기본값 그대로 실행하면 저장소 README에 보고한 추가 실험 결과를 다시 만듭니다. 데이터셋별 설정은 파트 F의 설정 셀(`F_DECAY_SCALED`, `F_PATIENCE_OVERRIDE`)에 있습니다. 처음부터 전부 돌리면 A100 기준 10시간 이상 걸리므로, `USE_DRIVE = True`로 결과를 Drive에 저장하며 나눠 실행하는 것을 권장합니다.

**논문**: Fechete et al., *Goal-Oriented Time-Series Forecasting: Foundation Framework Design*, [arXiv:2504.17493](https://arxiv.org/abs/2504.17493) · **공식 코드**: [netop-team/gotsf](https://github.com/netop-team/gotsf)

메인 노트북(`GOTSF_Colab_Reproduction.ipynb`)에서 다루지 않은 실험을 재현합니다. 결과는 별도 폴더(`checkpoints_extra/`, `results_extra/`, `outputs_extra/`)에 저장되므로 메인 노트북 결과와 섞이지 않습니다.

| 파트 | 논문 | 내용 | 소요 시간 (A100급, 추정) |
|---|---|---|---|
| **C** | Fig. 2, E2E 정책 | SynthDS에서 6개 정책(E2E, C, D₄, D₈ 두 방식, B)을 시드 10개로 비교 | 30–40분 |
| **D** | Fig. 5 | 예측으로 기지국 sleep을 결정하는 에너지 절감 응용 | 5분 |
| **E** | Fig. 6–9 | 구간 수 L, patching 방식, decay rate ν, 최소 구간 폭 δ의 민감도 | 30–40분 |
| **F** | Table 1 | Traffic, Electricity, Weather | 데이터셋당 1–3시간 |

각 파트는 §3의 `RUN_…` 스위치로 켜고 끌 수 있습니다. 처음에는 `RUN_MODE = "quick"`(모든 학습 1 에폭)으로 전체를 점검한 뒤 `"full"`로 실행하세요. 파트 F는 오래 걸리므로 §3의 Google Drive 저장을 켜 두는 것을 권장합니다.

**논문에 적혀 있지 않아 이 노트북이 정한 것**

| 실험 | 논문에 없는 정보 | 이 노트북의 선택 |
|---|---|---|
| Fig. 2 / E2E | E2E 구현 방식, C 정책의 δ | 구간 하나([0.75, 1])로만 학습하는 이산 정책(분류 손실 끔), C는 `w_hat=0.2` |
| Fig. 5 | 활용률 정규화 방식, 대상 beam, patching 방식, 오차 정의 | 전체 최댓값으로 나눠 [0, 1]로 정규화, 100개 beam 평균, `max` patching |
| Fig. 6–9 | 학습 설정, 그림 6·8의 patching 방식 | 저장소 toy 스크립트 설정, 두 patching 방식을 모두 표시 |
| Table 1 (3개 데이터셋) | 모델 설정, 100개 채널 선택, 값 범위 하한, decay의 단위 | 논문 부록 설정(`F_PRESET`으로 전환 가능), 앞 100개 채널, 하한 0, decay 37 |

## 0. 런타임 확인
메뉴 **런타임 → 런타임 유형 변경**에서 GPU를 선택하세요. 파트 F는 GPU 메모리가 큰 런타임(A100 등)을 권장합니다.

In [ ]:
import os, sys, re, json, time, shutil, subprocess
from pathlib import Path
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
%matplotlib inline

gpu = subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout.strip()
print(gpu or "⚠️ GPU가 없습니다 → CPU로 실행됩니다 (매우 느림)")
print(f"Python {sys.version.split()[0]} | torch {torch.__version__} (CUDA={torch.cuda.is_available()}) | "
      f"numpy {np.__version__} | pandas {pd.__version__}")

## 1. 공식 저장소 클론 & 패키지
메인 노트북과 같은 커밋(`31b17e5`)을 씁니다.

In [ ]:
BASE_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
REPO_DIR = os.path.join(BASE_DIR, "gotsf")
COMMIT   = "31b17e55a0cb6f41bfe25230db3f81567efd58f3"

def sh(cmd, quiet=False):
    # 셸 명령 실행 (실패하면 출력과 함께 예외)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    if r.returncode != 0:
        raise RuntimeError(f"명령 실패: {cmd}\n{out[-3000:]}")
    if out and not quiet:
        print(out)

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    sh(f"git clone -q https://github.com/netop-team/gotsf.git {REPO_DIR}")
if COMMIT:
    sh(f"git -C {REPO_DIR} checkout -q {COMMIT}")
os.chdir(REPO_DIR)
sh("git log -1 --format='commit %h (%cd) %s' --date=short")
sh(f"{sys.executable} -m pip install -q --no-deps lion-pytorch einops", quiet=True)
print("✔ 작업 폴더:", os.getcwd())

## 2. 패치
메인 노트북의 Colab 호환 패치 6개에 더해, 이번 실험에 필요한 패치 4개를 적용합니다. 새 패치는 모두 **환경변수를 설정할 때만 동작**하고 기본값은 원본과 같아서, 메인 노트북의 결과에는 영향이 없습니다.

| # | 파일 | 수정 | 이유 |
|---|---|---|---|
| 1–6 | (메인 노트북과 동일) | NumPy 2.x, figure 정리, Reformer import, CPU 폴백, `map_location`, pandas 2.x | Colab 호환 |
| 7 | `script_train.py` | 시드를 `GOTSF_SEED`에서 읽음 (기본 2023) | Fig. 2의 시드 10개 반복 |
| 8 | `data_provider/data_loader.py` | 분할 비율을 `GOTSF_PERC_TRAIN`, `GOTSF_PERC_TEST`에서 읽음 (기본 0.7, 0.2) | 논문의 SynthDS·Traffic·Electricity·Weather는 66/17/17 |
| 9 | `data_provider/data_factory.py` | 검증·테스트 배치 크기를 `GOTSF_EVAL_BS`에서 읽음 (기본: 전체를 한 배치) | 큰 데이터셋에서 GPU 메모리 부족 방지 |
| 10 | `experiments/exp_long_term_forecasting_*.py` | 구간 텐서를 배치 크기에 맞춰 자르고, 예측을 `np.concatenate`로 모음 | 9번을 쓸 때 필요. 전체 배치일 때는 결과가 같음 |

In [ ]:
def patch(path, old, new, required=True):
    p = Path(path); s = p.read_text()
    if old in s:
        p.write_text(s.replace(old, new)); print(f"✔ 패치: {path}")
    elif new in s:
        print(f"· 이미 적용됨: {path}")
    elif required:
        raise RuntimeError(f"패치할 코드를 찾지 못했습니다: {path} (COMMIT 고정 권장)")

# 1–6: 메인 노트북과 같은 Colab 호환 패치
patch("utils/tools.py", "np.Inf", "np.inf")
patch("utils/tools.py", "plt.savefig(name, bbox_inches='tight')\n\n",
      "plt.savefig(name, bbox_inches='tight')\n    plt.close()\n\n")
patch("layers/SelfAttention_Family.py", "\nfrom reformer_pytorch import LSHSelfAttention\n",
      "\ntry:\n    from reformer_pytorch import LSHSelfAttention\nexcept ImportError:  # Reformer 계열 모델에서만 필요\n    LSHSelfAttention = None\n")
patch("experiments/exp_basic.py", "if self.args.use_gpu:", "if self.args.use_gpu and torch.cuda.is_available():")
for f in ["none", "uniform", "discrete"]:
    path = f"experiments/exp_long_term_forecasting_{f}.py"
    patch(path, "torch.load(best_model_path)", "torch.load(best_model_path, map_location=self.device)")
    patch(path, "torch.load(os.path.join(self.args.checkpoints + self.args.experiment_name, 'checkpoint.pth'))",
          "torch.load(os.path.join(self.args.checkpoints + self.args.experiment_name, 'checkpoint.pth'), map_location=self.device)")
patch("data_provider/data_loader.py", ".drop(['date'], 1)", ".drop(columns=['date'])")

# 7: 시드
patch("script_train.py", "fix_seed = 2023", 'fix_seed = int(os.environ.get("GOTSF_SEED", 2023))')
# 8: 분할 비율
patch("data_provider/data_loader.py", "perc_train=0.7, perc_test=0.2)",
      'perc_train=float(os.environ.get("GOTSF_PERC_TRAIN", 0.7)), perc_test=float(os.environ.get("GOTSF_PERC_TEST", 0.2)))')
# 9: 검증·테스트 배치 크기
p = Path("data_provider/data_factory.py"); s = p.read_text()
if "GOTSF_EVAL_BS" in s:
    print("· 이미 적용됨: data_provider/data_factory.py")
else:
    s, n = re.subn(r"^(\s*)(data_loader = DataLoader\()",
                   r'\1_ebs = int(os.environ.get("GOTSF_EVAL_BS", 0))\n\1if _ebs > 0 and flag in ("test", "val"):\n'
                   r'\1    batch_size, drop_last = min(len(data_set), _ebs), False\n\1\2', s, flags=re.M)
    assert n == 1, "data_factory.py에서 DataLoader 생성 위치를 찾지 못했습니다"
    p.write_text("import os\n" + s); print("✔ 패치: data_provider/data_factory.py")
# 10: 작은 배치 평가 지원
for f in ["uniform", "discrete"]:
    path = f"experiments/exp_long_term_forecasting_{f}.py"
    patch(path, "self.model(batch_x, deltas_l, deltas_u)",
          "self.model(batch_x, deltas_l[:batch_x.shape[0]], deltas_u[:batch_x.shape[0]])")
    patch(path, "batch_y_mark, deltas_l, deltas_u)", "batch_y_mark, deltas_l[:batch_x.shape[0]], deltas_u[:batch_x.shape[0]])")
    patch(path, "deltas_l.view(batch_size, 1, 1)", "deltas_l[:batch_size].view(batch_size, 1, 1)")
    patch(path, "deltas_u.view(batch_size, 1, 1)", "deltas_u[:batch_size].view(batch_size, 1, 1)")
for f in ["none", "uniform", "discrete"]:
    path = f"experiments/exp_long_term_forecasting_{f}.py"
    for v in ["preds", "trues", "preds_cls", "trues_cls"]:
        patch(path, f"{v} = np.array({v})", f"{v} = np.concatenate({v}, axis=0)", required=v in ("preds", "trues"))

chk = subprocess.run([sys.executable, "-c", "import script_stats, experiments.exp_long_term_forecasting_none, "
                      "experiments.exp_long_term_forecasting_uniform, experiments.exp_long_term_forecasting_discrete"],
                     capture_output=True, text=True)
print("✔ 공식 코드 import 확인" if chk.returncode == 0 else chk.stderr[-3000:])

## 3. 공통 설정

In [ ]:
NB_VERSION   = "final"       # 결과 zip 이름에 붙음
RUN_MODE     = "full"     # "quick": 모든 학습 1 에폭으로 점검 | "full": 논문 설정
RUN_FIG2     = True       # 파트 C: Fig. 2, E2E 정책
RUN_FIG5     = True       # 파트 D: Fig. 5, 에너지 절감
RUN_SENS     = True       # 파트 E: Fig. 6–9, 하이퍼파라미터 민감도
RUN_TABLE1   = True       # 파트 F: Table 1의 Traffic, Electricity, Weather
SKIP_IF_DONE = True       # 이미 학습·평가한 실험은 건너뜀
FAST_PATCH   = True       # 공식 patching과 같은 결과를 내는 벡터화 버전 (큰 데이터셋에서 필수)

TAG = "" if RUN_MODE == "full" else "_quick"
CKPT_DIR, RES_DIR, OUT_DIR, LOG_DIR = (f"./checkpoints_extra{TAG}/", f"./results_extra{TAG}/",
                                       f"./outputs_extra{TAG}/", "./logs_extra/")
DATA_DIR = "./dataset/extra/"
for d in (OUT_DIR, LOG_DIR, DATA_DIR):
    os.makedirs(d, exist_ok=True)

def ep(n):                                    # quick 모드에서는 모든 학습을 1 에폭으로
    return n if RUN_MODE == "full" else 1

SYN_ENV = {"GOTSF_PERC_TRAIN": "0.66", "GOTSF_PERC_TEST": "0.17"}   # 논문: SynthDS는 66/17/17 분할
print(f"모드 {RUN_MODE} | 파트 C {RUN_FIG2}, D {RUN_FIG5}, E {RUN_SENS}, F {RUN_TABLE1} | 저장 위치 {CKPT_DIR}, {RES_DIR}")

### (선택) Google Drive에 저장
파트 F처럼 오래 걸리는 실험은 세션이 끊길 수 있으니 `USE_DRIVE = True`를 권장합니다. 다시 실행하면 끝난 실험은 건너뜁니다.

In [ ]:
USE_DRIVE = False         # True: 결과를 Google Drive에 저장하고, 다시 실행하면 끝난 실험은 건너뜀 (오래 걸리는 실행에 권장)
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = "/content/drive/MyDrive/gotsf_colab_extra"
    for d in (CKPT_DIR, RES_DIR, OUT_DIR, LOG_DIR, DATA_DIR):
        local = os.path.normpath(d)
        target = os.path.join(DRIVE_ROOT, local)
        os.makedirs(target, exist_ok=True)
        if os.path.islink(local):
            continue
        if os.path.isdir(local):
            shutil.copytree(local, target, dirs_exist_ok=True); shutil.rmtree(local)
        os.makedirs(os.path.dirname(local) or ".", exist_ok=True)
        os.symlink(target, local)
    print("✔ Drive에 저장:", DRIVE_ROOT)

## 4. 공통 도구
학습은 `script_train.py`를, 평가는 공식 `script_stats.ModelEvaluator`를 별도 프로세스로 실행합니다. `FAST_PATCH = True`이면 patching만 같은 계산을 하는 벡터화 버전으로 바꿉니다. `max` 방식은 결과가 완전히 같고, `expectation`은 부동소수점 합산 순서만 다릅니다. 공식 코드와 다른 점은 두 가지입니다.
- 평가 구간이 학습한 sub-interval과 같을 때 원본 예측 배열을 덮어쓰지 않습니다. 공식 코드는 덮어쓰기 때문에, 바로 다음 patching이 바뀐 배열을 읽게 됩니다.
- `expectation`에서 분류 확률이 모두 0으로 언더플로된 원소는 NaN 대신 단순 평균을 씁니다. Electricity에서 실제로 발생합니다.

In [ ]:
SUB_ENV = dict(os.environ, TQDM_DISABLE="1", PYTHONWARNINGS="ignore", MPLBACKEND="Agg")

def run_train(args, env=None):
    name, epochs = args["experiment_name"], int(args["train_epochs"])
    if SKIP_IF_DONE and os.path.exists(os.path.join(CKPT_DIR, name, "train_loss.npy")):
        print(f"⏭  {name}: 이미 학습됨"); return
    log_path = os.path.join(LOG_DIR, f"train_{name}{TAG}.log")
    cmd = [sys.executable, "-u", "script_train.py"] + [x for k, v in args.items() for x in (f"--{k}", str(v))]
    t0, last = time.time(), None
    with open(log_path, "w", buffering=1) as log:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env={**SUB_ENV, **(env or {})})
        try:
            for line in p.stdout:
                log.write(line)
                m = re.match(r"Epoch: (\d+), Steps: \d+ \| (.*)", line)
                if m:
                    last = (int(m.group(1)), m.group(2).strip())
            p.wait()
        except BaseException:          # 셀을 중단하면 학습 프로세스도 함께 종료 (GPU를 계속 점유하지 않도록)
            p.kill(); p.wait(); raise
    if p.returncode != 0:
        print("".join(open(log_path).readlines()[-30:]))
        raise RuntimeError(f"{name} 학습 실패 → 로그: {log_path}")
    print(f"✅ {name} ({time.time() - t0:.0f}s, {last[0] if last else '?'}/{epochs} ep | {last[1] if last else ''})")

EVAL_SCRIPT = '''
# stats.sh와 같은 평가를 한 프로세스에서 수행 (공식 script_stats.ModelEvaluator 사용)
import argparse, json, os, random, sys, time
import numpy as np, torch
sys.path.insert(0, os.getcwd())
from script_stats import ModelEvaluator
from utils.tools import get_interval_cover
from utils.metrics import metric

def fast_patch(self):
    # 공식 patch_experiments_discrete와 같은 계산을 벡터화 (max: 결과 동일, expectation: 합산 순서만 다름)
    l, u = self.args.eval_interval_l, self.args.eval_interval_u
    cover = get_interval_cover(train_discrete_intervals=self.train_discrete_intervals, eval_delta_l=l, eval_delta_u=u)
    # 경계값의 부동소수점 오차로 ROI와 사실상 겹치지 않는 구간이 끼어든 경우만 제외 (정상적인 경우 결과 동일)
    cover = [iv for iv in cover if min(iv[1], u) - max(iv[0], l) > 1e-9 * max(1.0, abs(u - l))]
    data = [self.load_data_discrete(iv) for iv in cover]
    P = np.stack([d[0] for d in data]); T = np.stack([d[1] for d in data]); C = np.stack([d[2] for d in data])
    if self.args.strategy == "max":
        k = np.argmax(C, axis=0)[None]
        combined_preds = np.take_along_axis(P, k, 0)[0]
        combined_trues = np.take_along_axis(T, k, 0)[0]
    else:
        csum = C.sum(0)
        zero = csum == 0                     # 분류 확률이 모두 0으로 언더플로된 원소 (공식 코드는 0/0 = NaN)
        combined_preds = ((C * P).sum(0) / np.where(zero, 1, csum)).astype(P.dtype)
        combined_preds[zero] = P.mean(0)[zero]   # → 단순 평균으로 대체
        combined_trues = T[0].copy()
        if zero.any():
            print(f"  expectation: 확률 합이 0인 원소 {zero.mean():.3%} → 단순 평균으로 대체")
    base = self.args.results_dir + "/" + self.args.experiment_name + "/" + f"({l}, {u})"
    mask = (combined_trues >= l) & (combined_trues <= u)
    m = np.array(metric(combined_preds * mask, combined_trues * mask))
    # ROI가 학습한 sub-interval과 같으면 그 폴더에는 원본 예측(pred_cls.npy 포함)이 있으므로 덮어쓰지 않음
    folders = [base + f"_{self.args.strategy}/"] + ([] if os.path.exists(base + "/pred_cls.npy") else [base + "/"])
    for folder in folders:
        os.makedirs(folder, exist_ok=True)
        np.save(folder + "pred.npy", combined_preds); np.save(folder + "true.npy", combined_trues)
        np.save(folder + "metrics.npy", m)
    print("Combined predictions - mse:{}, mae:{} (cover {})".format(m[1], m[0], len(cover)))

if os.environ.get("GOTSF_FAST_PATCH", "1") == "1":
    ModelEvaluator.patch_experiments_discrete = fast_patch

p = argparse.ArgumentParser()
p.add_argument("--experiments", nargs="+", required=True)
p.add_argument("--rois", required=True)
p.add_argument("--checkpoints_dir", default="checkpoints")
p.add_argument("--results_dir", default="results")
a = p.parse_args()
rois = [(float(l), float(u)) for l, u in json.loads(a.rois)]
for name in a.experiments:
    t0 = time.time()
    random.seed(2023); np.random.seed(2023); torch.manual_seed(2023)
    ev = ModelEvaluator(argparse.Namespace(
        experiment_name=name, eval_interval_l=0.0, eval_interval_u=0.0, eval_on_train=False, strategy="max",
        checkpoints_dir=a.checkpoints_dir, results_dir=a.results_dir, use_multi_gpu=False, use_gpu=True))
    if ev.model_args.training_interval_technique == "interval-discrete":
        ev.run_experiment_discrete()
        for l, u in rois:
            for strategy in ("expectation", "max"):
                ev.args.eval_interval_l, ev.args.eval_interval_u, ev.args.strategy = l, u, strategy
                ev.patch_experiments_discrete()
    else:
        for l, u in rois:
            ev.args.eval_interval_l, ev.args.eval_interval_u = l, u
            ev.run_experiment_none_uniform()
    del ev
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if os.environ.get("GOTSF_PRUNE_ARRAYS") == "1":     # 큰 예측 배열은 지우고 지표(metrics.npy)만 남김
        for root, _, files in os.walk(os.path.join(a.results_dir, name)):
            for f in files:
                if f.endswith(".npy") and f != "metrics.npy":
                    os.remove(os.path.join(root, f))
    print(f"[eval] {name} ({time.time() - t0:.0f}s)", flush=True)
'''
Path("colab_eval_extra.py").write_text(EVAL_SCRIPT)

def separate_interval(interval, nr_intervals):
    # utils.tools.separate_interval과 동일 (경계값의 부동소수점 표현까지 같아야 결과 폴더 이름이 일치)
    out, eps = [], (interval[1] - interval[0]) / nr_intervals
    d1, d2 = interval[0], interval[0] + eps
    for _ in range(nr_intervals):
        out.append((d1, d2)); d1, d2 = d1 + eps, d2 + eps
    return out

def load_metric(name, roi, suffix, idx=0):
    # results/<실험>/(l, u)<suffix>/metrics.npy = [mae, mse, rmse, mape, mspe]
    f = os.path.join(RES_DIR, name, f"({roi[0]}, {roi[1]}){suffix}", "metrics.npy")
    return float(np.load(f)[idx]) if os.path.exists(f) else np.nan

def eval_done(name, discrete, rois):
    ckpt = os.path.join(CKPT_DIR, name, "checkpoint.pth")
    sufs = ["_max", "_expectation"] if discrete else [""]
    files = [os.path.join(RES_DIR, name, f"({l}, {u}){s}", "metrics.npy") for l, u in rois for s in sufs]
    return all(os.path.exists(f) and os.path.getmtime(f) >= os.path.getmtime(ckpt) for f in files)

def run_eval(todo, rois, env=None, label="eval"):
    # todo: [(실험명, 이산 정책 여부)]
    names = [n for n, disc in todo if os.path.exists(os.path.join(CKPT_DIR, n, "checkpoint.pth"))
             and not (SKIP_IF_DONE and eval_done(n, disc, rois))]
    print(f"[{label}] 평가할 실험 {len(names)}개 (전체 {len(todo)}개 중)")
    if not names:
        return
    log_path = os.path.join(LOG_DIR, f"{label}{TAG}.log")
    cmd = [sys.executable, "-u", "colab_eval_extra.py", "--experiments", *names, "--rois", json.dumps(rois),
           "--checkpoints_dir", CKPT_DIR, "--results_dir", RES_DIR]
    run_env = {**SUB_ENV, **(env or {}), "GOTSF_FAST_PATCH": "1" if FAST_PATCH else "0"}
    with open(log_path, "a", buffering=1) as log:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=run_env)
        try:
            for line in p.stdout:
                log.write(line)
                if line.startswith("[eval]"):
                    print("  ✅", line[7:].rstrip())
            p.wait()
        except BaseException:
            p.kill(); p.wait(); raise
    if p.returncode != 0:
        print("".join(open(log_path).readlines()[-30:]))
        raise RuntimeError(f"평가 실패 → 로그: {log_path}")

# toy(SynthDS) 공통 설정: 저장소 iTransformer_train.sh 기준, 논문처럼 MAE로 학습
TOY_NOISY, TOY_CLEAN = "toy_example_random_noise_0.05_N_1_T_3072.csv", "toy_example_random_noise_0.0_N_1_T_3072.csv"
TOY_ROIS = separate_interval([0.0, 1.0], 4)
TOY_BASE = dict(root_path="./dataset/wireless/", data="custom", features="M", seq_len=48, pred_len=24, e_layers=2,
                enc_in=1, dec_in=1, c_out=1, des="Exp", d_model=128, d_ff=128, itr=1, n_heads=4, batch_size=32,
                lradj="type3", learning_rate=1e-3, dropout=0.1, num_workers=0, loss="MAE",
                checkpoints=CKPT_DIR, results=RES_DIR)

def discrete_policy(L, nu=37, lo=0.0, hi=1.0, epochs=60, **extra):
    return dict(model_id="Discrete", training_interval_technique="interval-discrete", nr_intervals=L, decay_rate=nu,
                train_interval_l=lo, train_interval_u=hi, train_epochs=ep(epochs), **extra)

def uniform_policy(w_hat, lo=0.0, hi=1.0, epochs=50):
    return dict(model_id="Uniform", training_interval_technique="interval-uniform", w_hat=w_hat, decay_rate=1e12,
                train_interval_l=lo, train_interval_u=hi, train_epochs=ep(epochs))

def none_policy(epochs=30):
    return dict(model_id="None", training_interval_technique="interval-none", train_epochs=ep(epochs))

# ── 논문 Table 1과 비교하는 도구 (파트 C, F) ─────────────────────────────
PAPER_POLICIES = ["D_L", "D¹_2L", "D∞_2L", "B", "C_0.2"]
EXTRA_POLICY = "C_0.1"

def table1_spec(name_of):
    return {"D_L": (name_of("discrete4"), "_max"), "D¹_2L": (name_of("discrete8"), "_expectation"),
            "D∞_2L": (name_of("discrete8"), "_max"), "B": (name_of("none"), ""), "C_0.2": (name_of("uniform02"), ""),
            EXTRA_POLICY: (name_of("uniform"), "")}

def interval_table(spec, rois, avg_name, scale=1.0):
    df = pd.DataFrame({lab: [load_metric(n, r, suf) * scale for r in rois] for lab, (n, suf) in spec.items()},
                      index=[f"I{k + 1}" for k in range(len(rois))])
    df.loc[avg_name] = df.mean()
    return df

def paper_table(rows, avg_name):
    df = pd.DataFrame(rows, index=[f"I{k + 1}" for k in range(len(rows))], columns=PAPER_POLICIES)
    df.loc[avg_name] = df.mean()
    return df

def improvement(df):
    others = [c for c in PAPER_POLICIES if c != "B"]
    return ((df["B"] - df[others].min(axis=1)) / df["B"] * 100).clip(lower=0)

def compare_summary(pairs, avg_name, caption):
    cols = [(c, w) for c in PAPER_POLICIES for w in ("재현", "논문")] + [(EXTRA_POLICY, "재현")]
    rows = [[(ours if w == "재현" else paper).loc[avg_name, c] for c, w in cols]
            + [improvement(ours)[avg_name], improvement(paper)[avg_name]] for ours, paper in pairs.values()]
    df = pd.DataFrame(rows, index=pd.MultiIndex.from_tuples(list(pairs), names=["데이터셋", "모델"]),
                      columns=pd.MultiIndex.from_tuples(cols + [("개선율 %", "재현"), ("개선율 %", "논문")]))
    display(df.style.format("{:.2f}", na_rep="-").set_caption(caption))
    return df

def plot_vs_paper(pairs, title, fname):
    mt = {"D_L": r"$D_L$", "D¹_2L": r"$D^1_{2L}$", "D∞_2L": r"$D^\infty_{2L}$", "B": "B", "C_0.2": r"$C_{0.2}$"}
    colors = dict(zip(PAPER_POLICIES, ["#DD8452", "#8C8C3C", "#C44E52", "#4C72B0", "#55A868"]))
    fig, axes = plt.subplots(1, len(pairs), figsize=(4.3 * len(pairs), 3.9), squeeze=False)
    for ax, ((ds, model), (ours, paper)) in zip(axes[0], pairs.items()):
        xs = [ix for ix in ours.index if "–" not in ix]
        for c in PAPER_POLICIES:
            ax.plot(xs, ours.loc[xs, c], color=colors[c], marker="o", ms=3.5, lw=1.6, label=mt[c])
            ax.plot(xs, paper.loc[xs, c], color=colors[c], ls="--", lw=1, alpha=0.8)
        ax.set(yscale="log", title=f"{model}", xlabel="interval")
    axes[0][0].set_ylabel("MAE (paper scale, log)")
    axes[0][-1].legend(fontsize=7, title="solid = ours, dashed = paper", title_fontsize=7)
    plt.suptitle(title); plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, fname), dpi=150); plt.show()

def aligned_toy_samples(n_rows, split_test=0.17, seq_len=48):
    # 예측 구간이 '출력 가설' 구간과 정확히 겹치는 test 샘플 (데이터는 48 스텝 주기: 입력 24 + 출력 24)
    n_te = int(n_rows * split_test)
    border1 = n_rows - n_te - seq_len
    s0 = (24 - (border1 + seq_len)) % 48
    return border1, list(range(s0, n_te - 24 + 1, 48))
print("✔ 공통 도구 준비 완료")

---
# Part C. Fig. 2와 E2E 정책 (SynthDS, iTransformer)
논문 Fig. 2는 관심 구간 I = [0.75, 1]에서 6개 정책의 예측을 시드 10개로 비교합니다. 학습 설정은 저장소 toy 스크립트(iTransformer, `lr=1e-3`)와 같고, 논문처럼 MAE로 학습하며 66/17/17로 분할합니다.

| 그림 | 정책 | 구현 |
|---|---|---|
| (a) | E2E | 구간 하나([0.75, 1])로만 학습. 이산 정책 코드에 `nr_intervals=1`, 지시 함수(`decay_rate=1e10`), 분류 손실 끔(`cls_weight=0`) |
| (b) | C | `w_hat=0.2` |
| (c) | D₄ | 구간 4개, [0.75, 1]은 학습 구간 그대로 |
| (d), (e) | D₈ (∞ / 1) | 구간 8개로 학습한 뒤 2개를 `max` / `expectation`으로 patching |
| (f) | B | 일반 학습 |

In [ ]:
FIG2_SEEDS = [2023 + k for k in range(10)]          # 논문 Fig. 2: 시드 10개
FIG2_ROI = (0.75, 1.0)
FIG2_POLICIES = {
    "e2e":       discrete_policy(1, nu=1e10, lo=0.75, hi=1.0, epochs=60, cls_weight=0.0),
    "uniform02": uniform_policy(0.2),
    "discrete4": discrete_policy(4),
    "discrete8": discrete_policy(8),
    "none":      none_policy(),
}
FIG2_PANELS = [("(a) E2E", "e2e", "_max"), ("(b) C", "uniform02", ""), ("(c) D4", "discrete4", "_max"),
               ("(d) D8, max", "discrete8", "_max"), ("(e) D8, expectation", "discrete8", "_expectation"), ("(f) B", "none", "")]
PAPER_SYNTH_I4_ITR = {"(c) D4": 13.1, "(e) D8, expectation": 12.4, "(d) D8, max": 13.6, "(f) B": 33.7, "(b) C": 13.4}  # Table 1, I4

fig2_name = lambda policy, seed: f"fig2_{policy}_s{seed}"

def fig2_run():
    jobs = [(p, s) for s in FIG2_SEEDS for p in FIG2_POLICIES]
    for i, (policy, seed) in enumerate(jobs, 1):
        print(f"▶ [{i}/{len(jobs)}]", end=" ")
        run_train(dict(TOY_BASE, **FIG2_POLICIES[policy], model="iTransformer", data_path=TOY_NOISY,
                       experiment_name=fig2_name(policy, seed)), env={**SYN_ENV, "GOTSF_SEED": str(seed)})
    run_eval([(fig2_name(p, s), p != "none" and p != "uniform02") for p, s in jobs], [FIG2_ROI], SYN_ENV, "eval_fig2")

def fig2_report():
    rows = []
    for title, policy, suf in FIG2_PANELS:
        v = np.array([load_metric(fig2_name(policy, s), FIG2_ROI, suf) for s in FIG2_SEEDS]) * 1e3
        rows.append({"정책": title, "MAE ×10³ 평균": np.nanmean(v), "표준편차": np.nanstd(v), "시드 수": int(np.isfinite(v).sum()),
                     "논문 Table 1 (I4)": PAPER_SYNTH_I4_ITR.get(title, np.nan)})
    tab = pd.DataFrame(rows).set_index("정책")
    tab.to_csv(os.path.join(OUT_DIR, "fig2_mae_on_I.csv"))
    display(tab.style.format("{:.2f}", na_rep="-").set_caption("I = [0.75, 1]에서의 마스크 MAE ×10³ (iTransformer, SynthDS)"))

    toy = pd.read_csv(f"dataset/wireless/{TOY_NOISY}")["OT"].to_numpy()
    border1, samples = aligned_toy_samples(len(toy))
    true = np.load(os.path.join(RES_DIR, fig2_name("none", FIG2_SEEDS[0]), f"({FIG2_ROI[0]}, {FIG2_ROI[1]})", "true.npy"))
    in_roi = [s for s in samples if FIG2_ROI[0] <= true[s, :, 0].mean() <= FIG2_ROI[1]]
    s = in_roi[0] if in_roi else samples[0]
    xc, xf = np.arange(-48, 0), np.arange(24)
    fig, axes = plt.subplots(2, 3, figsize=(16, 7), sharey=True)
    for ax, (title, policy, suf) in zip(axes.ravel(), FIG2_PANELS):
        preds = []
        for seed in FIG2_SEEDS:
            f = os.path.join(RES_DIR, fig2_name(policy, seed), f"({FIG2_ROI[0]}, {FIG2_ROI[1]}){suf}", "pred.npy")
            if os.path.exists(f):
                preds.append(np.load(f)[s, :, 0])
        ax.axhspan(*FIG2_ROI, color="purple", alpha=0.08)
        ax.plot(xc, toy[border1 + s: border1 + s + 48], color="gray", lw=1, label="input")
        ax.plot(xf, true[s, :, 0], color="k", lw=1.8, label="ground truth")
        if preds:
            mu, sd = np.mean(preds, 0), np.std(preds, 0)
            ax.plot(xf, mu, "--", color="tab:blue", lw=2, label=f"mean forecast ({len(preds)} seeds)")
            ax.fill_between(xf, mu - sd, mu + sd, color="tab:blue", alpha=0.25)
        ax.set(title=title, xlabel="time step (0 = forecast start)")
    axes[0][0].legend(fontsize=7, loc="lower left"); axes[0][0].set_ylabel("value"); axes[1][0].set_ylabel("value")
    plt.suptitle("Fig. 2 reproduction: iTransformer on SynthDS, interval of interest I = [0.75, 1] (purple)")
    plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "fig2.png"), dpi=150); plt.show()

In [ ]:
if RUN_FIG2:
    t0 = time.time(); fig2_run(); fig2_report(); print(f"파트 C: {(time.time() - t0) / 60:.1f}분")
else:
    print("파트 C 건너뜀 (RUN_FIG2 = False)")

---
# Part D. Fig. 5 — 에너지 절감 응용 (BLW-TrafficDS, iTransformer)
예측한 활용률 û(t)가 임계값 u_th 이상이면 capacity cell을 켜고(S=1), 아니면 끕니다(S=0). 논문처럼 B 정책과 D⋆ 정책(L=4)을 비교하고, D⋆는 관심 구간 [0, 0.5]를 sub-interval 2개로 patching합니다. 24스텝 예측을 4번 이어 96시간을 평가하고, 임계값은 0–2.5%입니다.

**원본 데이터로 크기 복원 (시도했지만 불가)**: 저장소의 `wireless.csv`는 전체를 하나의 값으로 나눠(전체 평균 1) 원래 활용률을 알 수 없습니다. 논문이 공개한 Hugging Face 데이터셋의 DLPRB(PRB 활용률, %)에서 같은 beam을 찾아 크기를 되살리려 했지만, `tools/fig5_data_diagnostic.ipynb`로 확인한 결과 공개 데이터(4개 지표, 모든 기간)에는 `wireless.csv`와 같은 시계열이 없었습니다(상관계수 중앙값 최대 0.69, 하루 주기만 일치). 그래서 기본값은 원래 크기가 필요 없는 `beammax`만 씁니다.

**정규화 후보**: 논문은 "[0, 1]로 정규화한 DLPRB 활용률"이라고만 해서, 두 가지를 모두 학습합니다.
- `pct`: 원래 DLPRB(%)를 100으로 나눔 (원본 데이터 필요)
- `beammax`: beam마다 최댓값으로 나눔 (`wireless.csv`만으로 계산 가능)

**논문 그림과의 비교**: Fig. 5의 세 곡선은 논문 PDF의 벡터 경로에서 추출했습니다. 그중 "Perfect Forecast" 곡선은 모델과 무관하게 데이터만으로 정해지므로, 정규화·96시간 구간·대상(100개 beam 평균, beam 평균 활용률, 개별 beam)은 **이 곡선과 가장 가까운 조합**으로 고릅니다. 모델 결과를 보지 않고 고르므로 B와 D의 비교에는 영향을 주지 않습니다. 개별 beam까지 후보로 넣으면 우연히 맞는 beam이 나올 수 있어, 100개 beam 평균의 결과도 함께 보여 줍니다.

In [ ]:
FIG5_HF_REPO  = "netop/Beam-Level-Traffic-Timeseries-Dataset"
FIG5_HF_FILES = ["data/train/DLPRB_train_0w-5w.csv", "data/test/DLPRB_test_5w-6w.csv"]   # 시간 순서대로 이어 붙임
FIG5_HF_LOCAL = None              # (점검용) 같은 이름의 파일을 이 폴더에서 읽음
FIG5_NORMS    = ["beammax"]        # "pct"(원래 DLPRB / 100)는 공개 데이터와 일치하지 않아 기본값에서 제외
FIG5_STRATEGY = "max"             # D⋆의 patching 방식 ("max" 또는 "expectation")
FIG5_WINDOWS  = [0, 24, 48, 72, 96]   # 96시간 구간의 시작 test 샘플 후보
TH = np.linspace(0, 0.025, 51)    # u_th: 0–2.5%
E_ON, E_OFF = 1266.0, 320.0       # 논문 부록 B: capacity cell 소비 전력 (Wh, 1시간당)
# 논문 Fig. 5 (PDF 벡터 경로에서 추출, u_th 0–2.5%를 0.05% 간격으로). 단위: Active/Hour (%)
PAPER_FIG5 = {"u_th_percent": [0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0, 1.05, 1.1, 1.15, 1.2, 1.25, 1.3, 1.35, 1.4, 1.45, 1.5, 1.55, 1.6, 1.65, 1.7, 1.75, 1.8, 1.85, 1.9, 1.95, 2.0, 2.05, 2.1, 2.15, 2.2, 2.25, 2.3, 2.35, 2.4, 2.45, 2.5], "Perfect": [90.26, 100.11, 100.11, 100.11, 100.11, 100.11, 100.11, 100.11, 100.11, 100.11, 100.11, 100.11, 100.11, 100.11, 100.11, 100.11, 99.5, 98.46, 97.98, 97.98, 96.42, 96.42, 96.42, 95.99, 95.99, 95.99, 95.34, 93.34, 93.34, 93.34, 93.34, 93.34, 93.34, 92.3, 92.3, 92.3, 92.3, 92.3, 92.3, 92.3, 92.3, 92.3, 92.3, 91.78, 91.78, 90.26, 90.26, 90.26, 90.26, 90.26, 90.26], "D": [89.31, 97.12, 96.98, 96.94, 96.72, 96.59, 96.51, 96.29, 96.07, 95.9, 95.73, 95.68, 95.6, 95.47, 95.29, 95.12, 94.77, 94.6, 94.51, 94.34, 94.21, 94.03, 93.99, 93.69, 93.64, 93.38, 93.25, 93.08, 92.91, 92.82, 92.65, 92.34, 92.04, 92.04, 91.82, 91.78, 91.69, 91.43, 91.08, 90.91, 90.78, 90.52, 90.48, 90.39, 90.26, 90.09, 89.83, 89.65, 89.61, 89.44, 89.39], "B": [86.96, 91.56, 91.52, 91.43, 91.34, 91.34, 91.21, 91.13, 90.91, 90.78, 90.65, 90.52, 90.43, 90.3, 90.17, 90.09, 90.0, 89.87, 89.74, 89.61, 89.44, 89.39, 89.26, 89.22, 89.18, 89.0, 88.96, 88.83, 88.74, 88.7, 88.65, 88.61, 88.57, 88.44, 88.44, 88.35, 88.35, 88.26, 88.13, 88.05, 87.96, 87.87, 87.83, 87.74, 87.74, 87.66, 87.61, 87.48, 87.44, 87.31, 87.14]}
PAPER_ENERGY = {"B-Policy": 950.0, "D-Policy": 337.0}   # 논문 부록 B: 하루 에너지 오차 (Wh)

def load_dlprb():
    frames = []
    for f in FIG5_HF_FILES:
        if FIG5_HF_LOCAL:
            path = os.path.join(FIG5_HF_LOCAL, os.path.basename(f))
        else:
            from huggingface_hub import hf_hub_download
            path = hf_hub_download(FIG5_HF_REPO, f, repo_type="dataset")
        df = pd.read_csv(path)
        frames.append(df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")]))
    return pd.concat(frames, ignore_index=True)

def zs(a):
    return (a - a.mean(0)) / (a.std(0) + 1e-12)

def match_beams(W, H, min_overlap=400):
    # W: wireless.csv (T, 100), H: 원본 DLPRB (S, beams). 시간 어긋남(lag = H 시작 - W 시작)과 beam 대응을 상관계수로 찾음
    T, S = len(W), len(H)
    best = (-1.0, 0)
    for lag in range(-(T - min_overlap), S - min_overlap + 1):
        w0, h0 = max(0, -lag), max(0, lag); L = min(T - w0, S - h0)
        c = np.nanmax((zs(W[w0:w0 + L, :1]) * zs(H[h0:h0 + L])).mean(0))
        if c > best[0]:
            best = (float(c), lag)
    lag = best[1]; w0, h0 = max(0, -lag), max(0, lag); L = min(T - w0, S - h0)
    C = zs(W[w0:w0 + L]).T @ zs(H[h0:h0 + L]) / L
    idx = C.argmax(1); corr = C[np.arange(len(idx)), idx]
    scale = []
    for v, j in enumerate(idx):
        m = W[w0:w0 + L, v] > 1e-9
        scale.append(np.median(H[h0:h0 + L, j][m] / W[w0:w0 + L, v][m]) if m.any() else np.nan)
    return lag, idx, corr, np.array(scale), (w0, h0, L)

def fig5_prepare():
    W_df = pd.read_csv("dataset/wireless/wireless.csv"); cols = list(W_df.columns[1:]); W = W_df[cols].to_numpy(float)
    U = {}
    if "pct" in FIG5_NORMS:
        try:
            H = np.nan_to_num(load_dlprb().to_numpy(float))
            lag, idx, corr, scale, (w0, h0, L) = match_beams(W, H)
            raw = W * scale[None, :]
            resid = np.abs(raw[w0:w0 + L] - H[h0:h0 + L][:, idx]).max() / max(H.max(), 1e-12)
            print(f"  원본 DLPRB {H.shape} ↔ wireless.csv {W.shape}: 시간 차이 {lag}시간, 겹치는 구간 {L}시간")
            print(f"  beam 상관계수 최소 {corr.min():.4f} · 중앙값 {np.median(corr):.4f} | 복원 오차 최대 {resid:.2e} (원본 최댓값 대비)")
            if corr.min() < 0.99:
                print("  ⚠️ 일부 beam이 정확히 일치하지 않습니다. 'pct' 결과는 해석에 주의하세요.")
            U["pct"] = raw / 100.0
        except Exception as e:
            print(f"  ⚠️ 원본 DLPRB로 크기를 복원하지 못해 'pct'를 건너뜁니다: {type(e).__name__}: {e}")
    if "beammax" in FIG5_NORMS:
        U["beammax"] = W / W.max(0, keepdims=True)
    for norm, u in U.items():
        df = pd.DataFrame(u, columns=cols); df.insert(0, "date", W_df["date"])
        df.to_csv(f"dataset/wireless/wireless_u_{norm}.csv", index=False)
        print(f"✔ wireless_u_{norm}.csv: u 범위 [{u.min():.4f}, {u.max():.4f}], 중앙값 {np.median(u):.4f}")
    return list(U)

BLW_BASE = dict(root_path="./dataset/wireless/", data="custom", features="M",
                seq_len=96, pred_len=24, e_layers=2, enc_in=100, dec_in=100, c_out=100, des="Exp", d_model=128, d_ff=128,
                itr=1, n_heads=4, batch_size=32, lradj="type3", learning_rate=1e-4, dropout=0.1, num_workers=0,
                loss="MAE", model="iTransformer", checkpoints=CKPT_DIR, results=RES_DIR)
FIG5_POLICIES = {"none": none_policy(60), "discrete4": discrete_policy(4, nu=37, epochs=60)}

def fig5_run():
    norms = fig5_prepare()
    for norm in norms:
        for policy, pol in FIG5_POLICIES.items():
            run_train(dict(BLW_BASE, **pol, data_path=f"wireless_u_{norm}.csv", experiment_name=f"fig5_{norm}_{policy}"))
        run_eval([(f"fig5_{norm}_none", False), (f"fig5_{norm}_discrete4", True)], [(0.0, 0.5)], None, f"eval_fig5_{norm}")
    return norms

def active(u):                    # u: (96, k) → 임계값마다 켜져 있는 시간 비율(%)
    return np.array([(u >= th).mean() * 100 for th in TH])

def sleep_err_h(u, uT):           # 하루(24시간)당 sleep 시간 오차, 임계값·계열 평균
    s = lambda a: np.array([(a < th).sum(0) for th in TH])
    return np.abs(s(u) - s(uT)).mean() / 4

def fig5_report(norms):
    paper = {k: np.array(v) for k, v in PAPER_FIG5.items() if k != "u_th_percent"}
    load = lambda n, folder, f: np.load(os.path.join(RES_DIR, n, folder, f))
    cands, summary = [], []
    for norm in norms:
        true = load(f"fig5_{norm}_none", "(0.0, 0.5)", "true.npy")
        preds = {"B": load(f"fig5_{norm}_none", "(0.0, 0.5)", "pred.npy"),
                 "D": load(f"fig5_{norm}_discrete4", f"(0.0, 0.5)_{FIG5_STRATEGY}", "pred.npy")}
        for start in FIG5_WINDOWS:
            idx = [start + 24 * k for k in range(4)]
            if idx[-1] >= len(true):
                continue
            ser = lambda a: np.concatenate([a[s] for s in idx], 0)          # (96, beams)
            uT, uP = ser(true), {k: ser(v) for k, v in preds.items()}
            targets = {"100 beams (avg of decisions)": (uT, uP),
                       "mean utilization of 100 beams": (uT.mean(1, keepdims=True), {k: v.mean(1, keepdims=True) for k, v in uP.items()})}
            targets.update({f"beam {c}": (uT[:, [c]], {k: v[:, [c]] for k, v in uP.items()}) for c in range(uT.shape[1])})
            for lab, (t, p) in targets.items():
                rmse = float(np.sqrt(np.mean((active(t) - paper["Perfect"]) ** 2)))
                cands.append(dict(norm=norm, start=start, target=lab, rmse=rmse, t=t, p=p))
    cdf = pd.DataFrame([{k: c[k] for k in ("norm", "start", "target", "rmse")} for c in cands])
    print("Perfect Forecast 곡선과의 RMSE (%p, 낮을수록 논문과 비슷): 정규화별 최솟값")
    display(cdf.loc[cdf.groupby("norm")["rmse"].idxmin()].set_index("norm").round(3))
    best = min(cands, key=lambda c: c["rmse"])
    avg = min((c for c in cands if c["norm"] == best["norm"] and c["target"].startswith("100 beams")), key=lambda c: c["rmse"])
    rows = []
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.2), sharey=True)
    for ax, (title, c) in zip(axes, [("best match to paper's Perfect curve", best), ("100 beams (avg of decisions)", avg)]):
        x = TH * 100
        ax.plot(x, active(c["t"]), color="tab:blue", label="Perfect Forecast")
        ax.plot(x, active(c["p"]["D"]), color="tab:red", ls="--", label="D-Policy")
        ax.plot(x, active(c["p"]["B"]), color="tab:green", ls="-.", label="B-Policy")
        for k, col in (("Perfect", "tab:blue"), ("D", "tab:red"), ("B", "tab:green")):
            ax.plot(x, paper[k], color=col, lw=3, alpha=0.25)
        ax.set(title=f"{title}\n{c['norm']}, window {c['start']}, {c['target']} (Perfect RMSE {c['rmse']:.2f}%p)",
               xlabel="u_th (%)", ylabel="Active/Hour (%)")
        ax.legend(fontsize=8, title="thin = ours, thick faded = paper", title_fontsize=7)
        for name, key in (("B-Policy", "B"), ("D-Policy", "D")):
            e = sleep_err_h(c["p"][key], c["t"])
            rows.append({"대상": title, "정책": name, "sleep 시간 오차 (h/일)": e, "에너지 오차 (Wh/일)": e * (E_ON - E_OFF),
                         "논문 에너지 오차 (Wh/일)": PAPER_ENERGY[name],
                         "곡선 RMSE vs 논문 (%p)": float(np.sqrt(np.mean((active(c["p"][key]) - paper[key]) ** 2)))})
    plt.suptitle("Fig. 5 reproduction with original DLPRB scale"); plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "fig5_v6.png"), dpi=150); plt.show()
    tab = pd.DataFrame(rows).set_index(["대상", "정책"])
    tab.to_csv(os.path.join(OUT_DIR, "fig5_v6_energy.csv"))
    display(tab.style.format("{:.3f}").set_caption("에너지 오차: 논문은 D가 B의 약 1/3 (337 대 950 Wh/일)"))

In [ ]:
if RUN_FIG5:
    t0 = time.time(); norms = fig5_run(); fig5_report(norms); print(f"파트 D: {(time.time() - t0) / 60:.1f}분")
else:
    print("파트 D 건너뜀 (RUN_FIG5 = False)")

---
# Part E. Fig. 6–9 — 하이퍼파라미터 민감도 (SynthDS)

| 그림 | 모델 | 바꾸는 값 | 평가 |
|---|---|---|---|
| Fig. 6 | PatchTST, D_L | L ∈ {4, 8, 16, 32}, ν = 37 | 폭 1/32–1인 2의 거듭제곱 분할마다 patching한 평균 MAE |
| Fig. 7 | PatchTST, D_L, **노이즈 없는** SynthDS | L ∈ {4, 8, 16, 32} | 오차비 γ = MAE(∞) / MAE(1) |
| Fig. 8 | PatchTST, D₈ | ν ∈ {0, 1, 2, 3, 5, ∞} | Fig. 6과 같은 방식 |
| Fig. 9 | iTransformer, C | δ(`w_hat`) ∈ {0, 5, 10, 15, 20, 25, 35}% | 구간 4개의 평균 MAE |

논문은 Fig. 6, 8에서 어떤 patching 방식을 썼는지 밝히지 않아 두 방식을 모두 그립니다. Fig. 8은 캡션(D₈)과 본문(D₃₂)이 달라, 그림의 x축(1/8–1)과 맞는 D₈로 학습합니다.

In [ ]:
SENS_L = [4, 8, 16, 32]
SENS_NU = [0, 1, 2, 3, 5, 1e10]                        # 1e10 = ∞ (지시 함수)
SENS_DELTA = [0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.35]
WIDTHS = [1 / 32, 1 / 16, 1 / 8, 1 / 4, 1 / 2, 1.0]

def partition(w):
    return [(i * w, (i + 1) * w) for i in range(round(1 / w))]

def rois_for(L):
    return [r for w in WIDTHS if w >= 1 / L for r in partition(w)]

nu_label = lambda nu: "inf" if nu >= 1e9 else f"{nu:g}"

def sens_run():
    runs = ([(f"fig6_D{L}", "PatchTST", TOY_NOISY, discrete_policy(L)) for L in SENS_L]
            + [(f"fig7_D{L}", "PatchTST", TOY_CLEAN, discrete_policy(L)) for L in SENS_L]
            + [(f"fig8_nu{nu_label(nu)}", "PatchTST", TOY_NOISY, discrete_policy(8, nu=nu)) for nu in SENS_NU]
            + [(f"fig9_delta{d:g}", "iTransformer", TOY_NOISY, uniform_policy(d)) for d in SENS_DELTA])
    for i, (name, model, data, pol) in enumerate(runs, 1):
        print(f"▶ [{i}/{len(runs)}]", end=" ")
        run_train(dict(TOY_BASE, **pol, model=model, data_path=data, experiment_name=name), env=SYN_ENV)
    for L in SENS_L:
        run_eval([(f"fig6_D{L}", True), (f"fig7_D{L}", True)], rois_for(L), SYN_ENV, f"eval_fig67_D{L}")
    run_eval([(f"fig8_nu{nu_label(nu)}", True) for nu in SENS_NU], rois_for(8), SYN_ENV, "eval_fig8")
    run_eval([(f"fig9_delta{d:g}", False) for d in SENS_DELTA], TOY_ROIS, SYN_ENV, "eval_fig9")

def avg_mae(name, w, suffix):
    return np.mean([load_metric(name, r, suffix) for r in partition(w)])

def sens_report():
    fig, axes = plt.subplots(1, 2, figsize=(13, 3.8), sharey=True)
    for ax, (suf, lab) in zip(axes, [("_max", "max (inf-strategy)"), ("_expectation", "expectation (1-strategy)")]):
        for L in SENS_L:
            ws = [w for w in WIDTHS if w >= 1 / L]
            ax.plot(ws, [avg_mae(f"fig6_D{L}", w, suf) for w in ws], marker="o", label=f"L = {L}")
        ax.set(title=f"Fig. 6: PatchTST, {lab}", xlabel="interval width", ylabel="avg. MAE over intervals")
        ax.legend(fontsize=8)
    plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "fig6.png"), dpi=150); plt.show()

    fig, ax = plt.subplots(figsize=(6.5, 3.8))
    for L in SENS_L:
        ws = [w for w in WIDTHS if w > 1 / L and w >= 0.25]
        ax.plot(ws, [avg_mae(f"fig7_D{L}", w, "_max") / avg_mae(f"fig7_D{L}", w, "_expectation") for w in ws],
                marker="o", label=f"L = {L}")
    ax.axhline(1, color="k", ls="--", lw=1)
    ax.set(title="Fig. 7: error ratio gamma = MAE(inf) / MAE(1), noise-free SynthDS", xlabel="interval width", ylabel="gamma")
    ax.legend(fontsize=8); plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "fig7.png"), dpi=150); plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(13, 3.8), sharey=True)
    for ax, (suf, lab) in zip(axes, [("_max", "max (inf-strategy)"), ("_expectation", "expectation (1-strategy)")]):
        ws = [w for w in WIDTHS if w >= 1 / 8]
        for nu in SENS_NU:
            ax.plot(ws, [avg_mae(f"fig8_nu{nu_label(nu)}", w, suf) for w in ws], marker="o", label=f"nu = {nu_label(nu)}")
        ax.set(yscale="log", title=f"Fig. 8: PatchTST D8, {lab}", xlabel="interval width", ylabel="avg. MAE (log)")
        ax.legend(fontsize=8)
    plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "fig8.png"), dpi=150); plt.show()

    d9 = pd.DataFrame({"delta (%)": [d * 100 for d in SENS_DELTA],
                       "avg MAE x1e3": [np.mean([load_metric(f"fig9_delta{d:g}", r, "") for r in TOY_ROIS]) * 1e3
                                        for d in SENS_DELTA]})
    fig, ax = plt.subplots(figsize=(6.5, 3.8))
    ax.plot(d9["delta (%)"], d9["avg MAE x1e3"], marker="o")
    ax.set(title="Fig. 9: C-policy, iTransformer", xlabel="separation percentage delta (%)", ylabel="average MAE (x1e3)")
    plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "fig9.png"), dpi=150); plt.show()
    d9.to_csv(os.path.join(OUT_DIR, "fig9.csv"), index=False)
    print("논문 Fig. 9: δ ≈ 5%에서 가장 낮고 그 뒤로 커지는 경향 (Table 1의 C₀.₂ = 13.2)")

In [ ]:
if RUN_SENS:
    t0 = time.time(); sens_run(); sens_report(); print(f"파트 E: {(time.time() - t0) / 60:.1f}분")
else:
    print("파트 E 건너뜀 (RUN_SENS = False)")

---
# Part F. Table 1 — Traffic, Electricity, Weather
논문 부록 A의 설정을 따릅니다: 입력/예측 길이 168/48, 66/17/17 분할, MAE 학습, L=4(D₄, patching은 D₈), 채널은 앞 100개(Weather는 21개 전부), 값 범위 상한 Traffic 0.2 · Weather 500 · Electricity 10000.

**데이터**
- Traffic, Electricity: LSTNet 저장소(GitHub)에서 자동으로 받습니다. 표준 벤치마크(Autoformer 등)의 CSV와 값이 같고, 날짜 열도 같은 방식(2016-07-01 02:00부터 1시간 간격)으로 만듭니다.
- Weather: Time-Series-Library의 Hugging Face 사본(`thuml/Time-Series-Library`의 `weather/weather.csv`)에서 자동으로 받습니다. `/content/weather.csv`나 `/content/drive/MyDrive/weather.csv`에 파일이 있으면 그것을 먼저 씁니다.

**디스크 사용량**: 공식 평가 코드는 구간마다 예측 배열을 저장해서, 3개 데이터셋을 모두 돌리면 약 85GB가 쌓입니다. 비교표에는 지표만 필요하므로 `F_PRUNE = True`(기본값)이면 각 실험의 평가가 끝나는 즉시 배열을 지우고 `metrics.npy`만 남깁니다. 실험 하나를 평가하는 동안만 몇 GB를 씁니다.

**decay rate의 단위 (`F_DECAY_SCALED`)**: 논문은 ν=37을 "값 범위 [0, 1]을 4등분했을 때 이웃 구간 중앙에서 가중치 1%"로 설명합니다. 원 단위 그대로 37을 쓰면 값 범위가 [0, 0.2]인 Traffic에서는 이웃 구간 중앙의 가중치가 약 40%로 훨씬 부드러워집니다. `F_DECAY_SCALED = True`이면 37을 값 범위 상한으로 나눠 논문의 설명과 맞춥니다. `F_DECAY = 1e10`이면 지시 함수(ν = ∞)가 됩니다. decay를 바꾸면 D 정책만 다시 학습하고, 결과 이름에 `_nuscaled`나 `_nuinf`가 붙어 기존 결과와 섞이지 않습니다.

검증 실행에서 확인한 것: BLW와 Electricity는 원 단위 37이 사실상 지시 함수인데 D_L이 논문과 일치했습니다. Traffic은 원 단위 37이 매우 부드러워(이웃 구간 중앙 가중치 40%) +14% 차이가 났고, scaled(가중치 1%)로 +12%까지 줄었습니다.

**설정 고르기 (`F_PRESET`)**: 논문 부록의 설정(`appendix`: 3층, `d_model=256`, `lr=1e-3`, 50 에폭, patience 5)이 기본입니다. 무선 데이터에서는 부록이 아닌 `train.sh` 설정(`trainsh`)이 논문 표와 일치했으므로, 먼저 `F_POLICIES = ["none"]`로 B만 돌려 논문의 B 열과 비교하고, 크게 다르면 `F_PRESET = "trainsh"`로 바꿔 보는 것을 권장합니다.

In [ ]:
F_DATASETS = ["Traffic", "Electricity", "Weather"]
F_MODELS   = ["iTransformer", "DLinear", "PatchTST", "TimeMixer"]
F_POLICIES = ["none", "uniform", "uniform02", "discrete4", "discrete8"]   # ["none", "uniform", "uniform02", "discrete4", "discrete8"] (B, C_0.1, C_0.2, D_4, D_8)
F_PRESET   = "appendix"     # "appendix": 논문 부록 | "trainsh": 무선 실험의 train.sh
F_EVAL_BS  = 1024           # 검증·테스트 배치 크기 (0이면 원본처럼 전체를 한 배치. 메모리 부족 시 줄이기)
F_PRUNE    = True           # 평가가 끝난 실험의 예측 배열을 지우고 지표만 남김 (안 지우면 3개 데이터셋에 약 85GB)
F_DECAY    = 37             # D 정책의 decay rate. 1e10 = ∞(지시 함수)
# D 정책의 decay 단위 (데이터셋별). True: 37 / 값 범위 상한 = 논문의 설명(이웃 구간 중앙 가중치 1%), False: 원 단위 37
# Traffic에서 True가 논문과 일치, Electricity는 False(원 단위)로 D_L이 일치, Weather는 Traffic의 규칙을 그대로 적용
F_DECAY_SCALED = {"Traffic": True, "Electricity": False, "Weather": True}
# 조기 종료 덮어쓰기 (patience 50 = 사실상 조기 종료 없음). 키: (데이터셋, 모델, 정책) / (데이터셋, 정책) / (모델, 정책) / 정책
# Traffic: 일찍 멈춰 논문과 어긋났던 실험만(D₈ 3개 모델, PatchTST B, TimeMixer B·C). Weather: Traffic의 규칙(D₈ 전부)을 그대로 적용
F_PATIENCE_OVERRIDE = {
    ("Traffic", "DLinear", "discrete8"): 50, ("Traffic", "PatchTST", "discrete8"): 50, ("Traffic", "TimeMixer", "discrete8"): 50,
    ("Traffic", "PatchTST", "none"): 50, ("Traffic", "TimeMixer", "none"): 50, ("Traffic", "TimeMixer", "uniform02"): 50,
    ("Weather", "discrete8"): 50,
}
F_CHANNELS = 100
WEATHER_CANDIDATES = ["/content/weather.csv", "/content/drive/MyDrive/weather.csv"]

F_SPECS = {   # 값 범위 상한(논문 부록 A)과 논문 Table 1의 표시 배율
    "Traffic":     dict(file="traffic_100.csv", upper=0.2, scale=500.0, label="×500"),
    "Electricity": dict(file="electricity_100.csv", upper=10000.0, scale=0.1, label="×10⁻¹"),
    "Weather":     dict(file="weather.csv", upper=500.0, scale=2.0, label="×2"),
}
F_PRESETS = {
    "appendix": dict(e_layers=3, d_model=256, d_ff=256, learning_rate=1e-3, train_epochs=50, patience=5),
    "trainsh":  dict(e_layers=2, d_model=128, d_ff=128, learning_rate=1e-4, train_epochs=60, patience=25),
}
F_ENV = {"GOTSF_PERC_TRAIN": "0.66", "GOTSF_PERC_TEST": "0.17", "GOTSF_EVAL_BS": str(F_EVAL_BS),
         "GOTSF_PRUNE_ARRAYS": "1" if F_PRUNE else "0"}

def build_lstnet(name, out):
    url = f"https://raw.githubusercontent.com/laiguokun/multivariate-time-series-data/master/{name}/{name}.txt.gz"
    x = pd.read_csv(url, header=None).iloc[:, :F_CHANNELS]
    df = pd.DataFrame(x.to_numpy(), columns=[str(i) for i in range(F_CHANNELS - 1)] + ["OT"])
    df.insert(0, "date", pd.date_range("2016-07-01 02:00:00", periods=len(df), freq="h").strftime("%Y-%m-%d %H:%M:%S"))
    df.to_csv(out, index=False)

def build_weather(out):
    for cand in WEATHER_CANDIDATES:
        if os.path.exists(cand):
            df = pd.read_csv(cand)
            assert "date" in df.columns and "OT" in df.columns, f"{cand}에 date, OT 열이 필요합니다"
            df.to_csv(out, index=False); print(f"  {cand} 사용"); return
    try:                                   # Time-Series-Library의 Hugging Face 사본에서 자동으로 받음
        from huggingface_hub import hf_hub_download
        p = hf_hub_download("thuml/Time-Series-Library", "weather/weather.csv", repo_type="dataset")
        shutil.copy(p, out); print("  Hugging Face(thuml/Time-Series-Library)에서 받음"); return
    except Exception as e:
        raise FileNotFoundError(f"weather.csv를 받지 못했습니다 ({type(e).__name__}). Time-Series-Library의 데이터셋 링크 "
                                "(Google Drive 또는 Hugging Face)에서 weather.csv를 받아 /content/weather.csv에 올린 뒤 다시 실행하세요")

def f_prepare():
    ready = []
    for ds in F_DATASETS:
        out = os.path.join(DATA_DIR, F_SPECS[ds]["file"])
        try:
            if not os.path.exists(out):
                print(f"▶ {ds} 데이터 준비")
                build_weather(out) if ds == "Weather" else build_lstnet(ds.lower(), out)
            cols = pd.read_csv(out, nrows=1).columns
            F_SPECS[ds]["enc"] = len(cols) - 1
            n = sum(1 for _ in open(out)) - 1
            print(f"✔ {ds}: {n}행 × {len(cols) - 1}개 변수 ({out})")
            ready.append(ds)
        except Exception as e:
            print(f"⚠️ {ds} 데이터 준비 실패 → 건너뜀: {type(e).__name__}: {e}")
    return ready

def patience_for(ds, model, policy):
    for key in ((ds, model, policy), (ds, policy), (model, policy), policy):
        if key in F_PATIENCE_OVERRIDE:
            return F_PATIENCE_OVERRIDE[key]
    return None

def f_name(ds, model, policy):
    suffix = ""
    if policy.startswith("discrete"):                 # decay 설정이 기본값(37, 원 단위)과 다르면 이름에 표시 → 결과가 섞이지 않음
        suffix = "_nuscaled" if F_DECAY_SCALED.get(ds, False) else ("" if F_DECAY == 37 else f"_nu{'inf' if F_DECAY >= 1e9 else f'{F_DECAY:g}'}")
    if patience_for(ds, model, policy) is not None:    # patience를 바꾸면 이름에 표시 → 결과가 섞이지 않음
        suffix += f"_p{patience_for(ds, model, policy)}"
    return f"{ds}_{model}_{policy}_{F_PRESET}" + suffix

def f_args(ds, model, policy):
    sp, pr = F_SPECS[ds], dict(F_PRESETS[F_PRESET])
    epochs = pr.pop("train_epochs")
    if patience_for(ds, model, policy) is not None:
        pr["patience"] = patience_for(ds, model, policy)
    up = sp["upper"]
    nu = F_DECAY / up if F_DECAY_SCALED.get(ds, False) else F_DECAY
    pol = {"none": none_policy(epochs), "uniform": uniform_policy(0.1, 0.0, up, epochs),
           "uniform02": uniform_policy(0.2, 0.0, up, epochs),
           "discrete4": discrete_policy(4, nu, 0.0, up, epochs), "discrete8": discrete_policy(8, nu, 0.0, up, epochs)}[policy]
    return dict(root_path=DATA_DIR, data_path=sp["file"], data="custom", features="M", seq_len=168, pred_len=48,
                enc_in=sp["enc"], dec_in=sp["enc"], c_out=sp["enc"], des="Exp", itr=1, n_heads=4, batch_size=32,
                lradj="type3", dropout=0.1, num_workers=0, loss="MAE", checkpoints=CKPT_DIR, results=RES_DIR,
                model=model, experiment_name=f_name(ds, model, policy), **pr, **pol)

def f_run(datasets):
    for ds in datasets:
        jobs = [(m, p) for m in F_MODELS for p in F_POLICIES]
        for i, (model, policy) in enumerate(jobs, 1):
            print(f"▶ {ds} [{i}/{len(jobs)}]", end=" ")
            run_train(f_args(ds, model, policy), env=F_ENV)
        rois = separate_interval([0.0, F_SPECS[ds]["upper"]], 4)
        run_eval([(f_name(ds, m, p), p.startswith("discrete")) for m, p in jobs], rois, F_ENV, f"eval_table1_{ds}")

# 논문 Table 1 (표시 배율 적용 값). 행 I1…I4, 열 D_L, D¹_2L, D∞_2L, B, C_0.2
PAPER_T1_OTHER = {
    "Traffic": {
        "DLinear":      [[1.70, 1.53, 1.67, 1.79, 1.82], [1.65, 1.35, 1.44, 1.99, 1.71], [0.61, 0.44, 0.47, 0.81, 0.62], [0.45, 0.31, 0.33, 0.59, 0.45]],
        "TimeMixer":    [[1.90, 1.71, 1.73, 2.44, 2.17], [1.25, 1.23, 1.27, 2.02, 1.52], [0.39, 0.36, 0.41, 0.74, 0.51], [0.22, 0.20, 0.24, 0.59, 0.35]],
        "PatchTST":     [[1.08, 1.02, 1.06, 1.41, 1.11], [0.91, 0.90, 0.93, 1.42, 0.89], [0.32, 0.30, 0.35, 0.65, 0.30], [0.16, 0.16, 0.19, 0.53, 0.16]],
        "iTransformer": [[1.29, 1.30, 1.32, 1.41, 1.36], [1.19, 1.22, 1.24, 1.50, 1.13], [0.53, 0.54, 0.55, 0.66, 0.53], [0.35, 0.36, 0.37, 0.51, 0.35]]},
    "Weather": {
        "DLinear":      [[1.05, 0.76, 0.75, 0.78, 1.32], [0.36, 0.23, 0.29, 0.42, 0.36], [0.13, 0.09, 0.10, 0.20, 0.26], [0.16, 0.14, 0.21, 0.21, 0.27]],
        "TimeMixer":    [[2.25, 0.73, 0.71, 1.00, 1.94], [0.37, 0.22, 0.26, 0.54, 0.43], [0.19, 0.08, 0.09, 0.26, 0.25], [0.17, 0.12, 0.13, 0.27, 0.26]],
        "PatchTST":     [[0.40, 0.67, 0.53, 0.65, 1.17], [0.22, 0.23, 0.28, 0.37, 0.22], [0.08, 0.09, 0.11, 0.22, 0.08], [0.10, 0.13, 0.14, 0.23, 0.11]],
        "iTransformer": [[0.49, 0.48, 0.49, 0.68, 1.72], [0.32, 0.32, 0.32, 0.42, 0.49], [0.18, 0.17, 0.18, 0.23, 0.39], [0.19, 0.19, 0.19, 0.21, 0.38]]},
    "Electricity": {
        "DLinear":      [[3.81, 3.82, 3.82, 3.77, 3.95], [1.15, 1.16, 1.16, 1.17, 1.14], [0.916, 0.929, 0.932, 0.955, 0.890], [0.369, 0.379, 0.380, 0.399, 0.353]],
        "TimeMixer":    [[3.83, 3.95, 3.95, 4.50, 4.86], [1.11, 1.07, 1.09, 1.41, 1.37], [0.807, 0.782, 0.836, 1.08, 0.984], [0.297, 0.277, 0.288, 0.449, 0.388]],
        "PatchTST":     [[3.33, 4.66, 4.69, 3.32, 3.78], [0.919, 1.24, 1.34, 1.01, 0.990], [0.671, 0.852, 0.931, 0.835, 0.689], [0.238, 0.248, 0.272, 0.370, 0.243]],
        "iTransformer": [[12.8, 10.3, 10.5, 3.26, 3.67], [4.67, 3.69, 3.84, 1.05, 1.02], [2.08, 1.84, 1.98, 0.861, 0.822], [0.567, 0.509, 0.522, 0.372, 0.322]]},
}

def f_report(datasets):
    for ds in datasets:
        sp, rois = F_SPECS[ds], separate_interval([0.0, F_SPECS[ds]["upper"]], 4)
        pairs = {(ds, m): (interval_table(table1_spec(lambda p: f_name(ds, m, p)), rois, "I1–I4", sp["scale"]),
                           paper_table(PAPER_T1_OTHER[ds][m], "I1–I4")) for m in F_MODELS}
        for (_, m), (ours, _) in pairs.items():
            ours.to_csv(os.path.join(OUT_DIR, f"table1_{ds}_{m}_{F_PRESET}.csv"))
        compare_summary(pairs, "I1–I4", f"{ds} ({sp['label']}): I1–I4 평균 MAE, 재현 vs 논문 Table 1 | 설정 {F_PRESET}")
        plot_vs_paper(pairs, f"{ds} ({sp['label']}), Table 1 protocol, preset = {F_PRESET}", f"table1_{ds}_{F_PRESET}.png")

In [ ]:
if RUN_TABLE1:
    t0 = time.time()
    ready = f_prepare()
    f_run(ready); f_report(ready)
    print(f"파트 F: {(time.time() - t0) / 60:.1f}분")
else:
    print("파트 F 건너뜀 (RUN_TABLE1 = False)")

---
## 결과 내려받기 (선택)
표(CSV)·그래프(PNG)·로그를 zip으로 묶습니다. 가중치까지 포함하려면 `INCLUDE_ARRAYS = True`로 바꾸세요(크기가 커질 수 있음).

In [ ]:
INCLUDE_ARRAYS = False
import zipfile
zip_path = os.path.join(BASE_DIR, f"gotsf_colab_extra_outputs_{NB_VERSION}{TAG}.zip")
folders = [OUT_DIR, LOG_DIR] + ([CKPT_DIR, RES_DIR] if INCLUDE_ARRAYS else [])
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in folders:
        for root, _, files in os.walk(os.path.normpath(folder), followlinks=True):
            for f in files:
                zf.write(os.path.join(root, f))
print(f"✔ {zip_path} ({os.path.getsize(zip_path) / 1e6:.1f} MB)")
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    pass